# Stochastic Calculus, Step by Step · Lesson 22: Checking the formula numerically

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Where a lesson needs the share's real-world expected return, it is μ = 8%. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. Monte Carlo and its error bar · 3. Does the error bar tell the truth? · 4. The binomial tree converges, unevenly · 5. Taming the oscillation · 6. The American put: what the formula can't do · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 22,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
mu = 0.08                         # the share's real-world expected return
theta_mpr = (mu - r) / sigma      # the market price of risk
Nv = np.vectorize(N)              # N, for arrays
phi = lambda x: np.exp(-np.asarray(x, float) ** 2 / 2) / math.sqrt(2 * math.pi)   # the normal density


def d_plus(s, tau, sig=sigma, k=K):
    return (np.log(s / k) + (r + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))


def d_minus(s, tau, sig=sigma, k=K):
    return d_plus(s, tau, sig, k) - sig * np.sqrt(tau)


def call(s, tau, sig=sigma, k=K):
    """The Black-Scholes call price: Lesson 20 derives it."""
    return s * Nv(d_plus(s, tau, sig, k)) - k * np.exp(-r * tau) * Nv(d_minus(s, tau, sig, k))


def put(s, tau, sig=sigma, k=K):
    return call(s, tau, sig, k) - s + k * np.exp(-r * tau)


out["bs"] = {"mu": mu, "theta": theta_mpr, "d_plus": float(d_plus(S0, T)), "d_minus": float(d_minus(S0, T)),
             "N_plus": N(float(d_plus(S0, T))), "N_minus": N(float(d_minus(S0, T))),
             "call": float(call(S0, T)), "put": float(put(S0, T))}
out["bs"]

## 2. Monte Carlo and its error bar

Draw $M$ risk-neutral prices at $T$, average the discounted payoffs. The average is random, with standard error $s/\sqrt M$ where $s$ is the standard deviation of one discounted payoff. Each hundred-fold increase in paths buys one more digit.

In [ ]:
rng = np.random.default_rng(seed)
bs_c, disc = float(call(S0, T)), math.exp(-r * T)


def mc_call(n_paths, gen, k=K):
    Z = gen.standard_normal(n_paths)
    pay = disc * np.maximum(S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * Z) - k, 0)
    return float(pay.mean()), float(pay.std(ddof=1) / math.sqrt(n_paths)), float(pay.std(ddof=1))


mc_rows = []
for m in (1_000, 10_000, 100_000, 1_000_000, 10_000_000):
    est, se, s_ = mc_call(m, rng)
    mc_rows.append({"M": m, "est": est, "se": se, "low": est - 1.96 * se, "high": est + 1.96 * se, "error": est - bs_c,
                    "inside": "yes" if est - 1.96 * se < bs_c < est + 1.96 * se else "no"})
out["mc"] = {"rows": mc_rows, "bs": bs_c, "s": s_, "paths_for_cent": (1.96 * s_ / 0.01) ** 2}
pd.DataFrame(mc_rows)

## 3. Does the error bar tell the truth?

A 95% confidence interval should contain the true price 95% of the time. Run the whole experiment 2,000 times with 10,000 paths each and count. The spread of the 2,000 estimates should also match the $s/\sqrt M$ that a single run reports.

In [ ]:
trials, m_ = 2000, 10_000
ests = np.empty(trials); ses = np.empty(trials)
for i in range(trials):
    ests[i], ses[i], _ = mc_call(m_, rng)
inside = (ests - 1.96 * ses < bs_c) & (bs_c < ests + 1.96 * ses)
out["cover"] = {"trials": trials, "M": m_, "coverage": float(inside.mean()), "sd_est": float(ests.std(ddof=1)),
                "mean_se": float(ses.mean()), "mean_est": float(ests.mean()), "bias": float(ests.mean() - bs_c)}
fig_m = [10 ** k for k in (2, 3, 4, 5, 6)]
fig_err = []
for m in fig_m:
    fig_err.append(float(np.sqrt(np.mean([(mc_call(m, rng)[0] - bs_c) ** 2 for _ in range(60 if m < 10**6 else 12)]))))
out["cover"]["fig_m"], out["cover"]["fig_err"] = fig_m, fig_err
out["cover"]["s_"] = s_
{k_: v for k_, v in out["cover"].items() if not k_.startswith("fig")}

## 4. The binomial tree converges, unevenly

The risk-neutral tree of Part 2, with $u = e^{\sigma\sqrt{\Delta t}}$, $d = 1/u$ (the Cox-Ross-Rubinstein choice), reaches the formula as the steps shrink. But the error jumps up and down as the steps go from odd to even: the strike falls in a different place among the final nodes each time.

In [ ]:
def crr(n_steps, payoff, american=False, s0=S0, sig=sigma, t_=T):
    dt = t_ / n_steps
    u_ = math.exp(sig * math.sqrt(dt)); d_ = 1 / u_; p_ = (math.exp(r * dt) - d_) / (u_ - d_); df = math.exp(-r * dt)
    s_end = s0 * u_ ** (2 * np.arange(n_steps + 1) - n_steps)
    v = payoff(s_end)
    for i in range(n_steps - 1, -1, -1):
        v = df * (p_ * v[1:] + (1 - p_) * v[:-1])
        if american:
            v = np.maximum(v, payoff(s0 * u_ ** (2 * np.arange(i + 1) - i)))
    return float(v[0])


call_pay = lambda s: np.maximum(s - K, 0)
put_pay = lambda s: np.maximum(K - s, 0)
tree_rows = [{"n": n_, "price": crr(n_, call_pay), "error": crr(n_, call_pay) - bs_c} for n_ in (10, 11, 50, 51, 100, 101, 500, 501, 1000, 1001)]
scan_n = list(range(10, 201))
scan_err = [crr(n_, call_pay) - bs_c for n_ in scan_n]
out["tree"] = {"rows": tree_rows, "scan_n": scan_n, "scan_err": scan_err}
pd.DataFrame(tree_rows)

## 5. Taming the oscillation

A single tree size can be lucky, so judge a method by its **worst** error over twenty neighbouring sizes. Two cures: average two neighbouring sizes, or smooth the last step by using the Black-Scholes value at the nodes one step before expiry, so that the payoff's kink is never placed on the grid.

In [ ]:
def crr_smooth(n_steps):
    # The same tree, but the last step uses the formula at the nodes one step before expiry.
    dt = T / n_steps
    u_ = math.exp(sigma * math.sqrt(dt)); d_ = 1 / u_; p_ = (math.exp(r * dt) - d_) / (u_ - d_); df = math.exp(-r * dt)
    v = call(S0 * u_ ** (2 * np.arange(n_steps) - (n_steps - 1)), dt)
    for i in range(n_steps - 2, -1, -1):
        v = df * (p_ * v[1:] + (1 - p_) * v[:-1])
    return float(v[0])


avg_rows = []
for n_ in (25, 50, 100, 200, 400, 800):
    win = range(n_, n_ + 20)
    plain = [crr(j, call_pay) - bs_c for j in win]
    crr_next = plain[1:] + [crr(n_ + 20, call_pay) - bs_c]
    avg_rows.append({"n": n_, "plain": max(abs(x) for x in plain), "avg": max(abs(0.5 * (a_ + b_)) for a_, b_ in zip(plain, crr_next)),
                     "smooth": max(abs(crr_smooth(j) - bs_c) for j in win)})
scan_smooth = [crr_smooth(n_) - bs_c for n_ in scan_n]
out["avg"] = {"rows": avg_rows, "scan_smooth": scan_smooth, "ratio": avg_rows[-2]["smooth"] / avg_rows[-1]["smooth"]}
pd.DataFrame(avg_rows)

## 6. The American put: what the formula can't do

There is no Black-Scholes formula for the American put, but the tree handles it in one line: at each node take the larger of holding and exercising. The result stands above the European put, and settles as the steps grow.

In [ ]:
bs_p = float(put(S0, T))
am_rows = []
for n_ in (50, 100, 200, 400, 800, 1600):
    e_ = 0.5 * (crr(n_, put_pay) + crr(n_ + 1, put_pay)); a_ = 0.5 * (crr(n_, put_pay, True) + crr(n_ + 1, put_pay, True))
    am_rows.append({"n": n_, "european": e_, "american": a_, "premium": a_ - e_})
out["am"] = {"rows": am_rows, "bs_put": bs_p, "best": am_rows[-1]["american"], "premium": am_rows[-1]["american"] - bs_p}
pd.DataFrame(am_rows)

In [ ]:
# Chart data for the video (no random numbers)
out["tree"]["ticks"] = [{"x": n_, "label": str(n_)} for n_ in (10, 50, 100, 150, 200)]

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 4: Solutions*.

**Exercise 1.** One discounted call payoff has a standard deviation of about 12.5. How many paths do you need for a 95% confidence half-width of 0.01? Of 0.001? Why is the second answer 100 times the first, not 10?

**Exercise 2.** **Antithetic variates.** Estimate with pairs $Z$ and $-Z$: $\hat V = \tfrac12\big(f(Z) + f(-Z)\big)$. Show that its variance is $\tfrac12\mathrm{Var}(f(Z))(1 + \rho)$, where $\rho$ is the correlation between $f(Z)$ and $f(-Z)$. Why does this help for a monotone payoff?

**Exercise 3.** Explain why a call priced on a binomial tree jumps up and down as the steps increase, by looking at where the strike falls among the final nodes. Why is a put on the same tree no better, and why does a smooth payoff not have the problem?

**Exercise 4.** **Richardson extrapolation.** If a method's error is $c/n$ for some unknown $c$, show that $2V_{2n} - V_n$ cancels it. With $V_{100} = 9.0330$ and $V_{200} = 9.0292$ (averaged-neighbour prices of the European put), what is the extrapolated value, and how does it compare with the formula's 9.0248?

**Exercise 5.** **Control variate.** You know $E[e^{-rT}S_T] = S_0$ exactly. Show that $\hat V = \bar Y - b(\bar X - S_0)$, with $Y$ the discounted payoff and $X = e^{-rT}S_T$, is unbiased for any $b$; that the best $b$ is $\mathrm{Cov}(Y, X)/\mathrm{Var}(X)$; and that the variance falls by the factor $1 - \rho^2$.

**Exercise 6.** A cash digital pays 1 if $S_T > K$, with $\tilde P(S_T > K) = 0.4231$. What is the standard deviation of one discounted payoff? How many paths give a 95% half-width of 0.001, and how does the answer change for a strike so high that the probability is 0.01?

**Exercise 7.** (Code) Add antithetic variates and then a control variate to the call's Monte Carlo with $10^5$ paths. Report each estimator's standard error, and the number of plain paths each one is worth. Does the control variate help more for $K = 85$ or $K = 125$, and why?

**Exercise 8.** (Code) Price the American put with $n$ of 50, 100, 200, 400, 800 steps, using neighbour averaging, then Richardson extrapolation. Take $n = 6400$ as the benchmark: how close does each method get at each $n$? Find the share price below which immediate exercise is best with one year left.

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")